In [0]:
# import libraries / functions
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, when

spark = SparkSession.builder.appName("BronzeBabyNames").getOrCreate()

# Load your Bronze table
df = spark.table("bronze_baby_names")

df.printSchema()
df.show(8)

root
 |-- name: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- count: integer (nullable = true)
 |-- source_file: string (nullable = true)

+--------+---+-----+--------------------+
|    name|sex|count|         source_file|
+--------+---+-----+--------------------+
|   Emily|  F|19367|dbfs:/Volumes/wor...|
|Isabella|  F|19143|dbfs:/Volumes/wor...|
|    Emma|  F|18390|dbfs:/Volumes/wor...|
|     Ava|  F|18053|dbfs:/Volumes/wor...|
| Madison|  F|17968|dbfs:/Volumes/wor...|
|  Sophia|  F|17031|dbfs:/Volumes/wor...|
|  Olivia|  F|16589|dbfs:/Volumes/wor...|
| Abigail|  F|15476|dbfs:/Volumes/wor...|
+--------+---+-----+--------------------+
only showing top 8 rows


Narrow transformations process data within the same partition and are efficient as they do not require data movement across nodes. (they are not costly)

In [0]:
# filter is a narrow transformation
df_filtered = df.filter(col("count") > 100)

df_filtered.show()

+---------+---+-----+--------------------+
|     name|sex|count|         source_file|
+---------+---+-----+--------------------+
|    Emily|  F|19367|dbfs:/Volumes/wor...|
| Isabella|  F|19143|dbfs:/Volumes/wor...|
|     Emma|  F|18390|dbfs:/Volumes/wor...|
|      Ava|  F|18053|dbfs:/Volumes/wor...|
|  Madison|  F|17968|dbfs:/Volumes/wor...|
|   Sophia|  F|17031|dbfs:/Volumes/wor...|
|   Olivia|  F|16589|dbfs:/Volumes/wor...|
|  Abigail|  F|15476|dbfs:/Volumes/wor...|
|   Hannah|  F|13323|dbfs:/Volumes/wor...|
|Elizabeth|  F|13071|dbfs:/Volumes/wor...|
|  Addison|  F|11949|dbfs:/Volumes/wor...|
| Samantha|  F|11883|dbfs:/Volumes/wor...|
|   Ashley|  F|11428|dbfs:/Volumes/wor...|
|   Alyssa|  F|11275|dbfs:/Volumes/wor...|
|      Mia|  F|10922|dbfs:/Volumes/wor...|
|    Chloe|  F|10759|dbfs:/Volumes/wor...|
|  Natalie|  F|10443|dbfs:/Volumes/wor...|
|    Sarah|  F|10014|dbfs:/Volumes/wor...|
|   Alexis|  F| 9923|dbfs:/Volumes/wor...|
|    Grace|  F| 9769|dbfs:/Volumes/wor...|
+---------+

In [0]:
# add column is a narrow transformation
df_added = df_filtered.withColumn("name_upper", upper(col("name")))

df_added.show()

+---------+---+-----+--------------------+----------+
|     name|sex|count|         source_file|name_upper|
+---------+---+-----+--------------------+----------+
|    Emily|  F|19367|dbfs:/Volumes/wor...|     EMILY|
| Isabella|  F|19143|dbfs:/Volumes/wor...|  ISABELLA|
|     Emma|  F|18390|dbfs:/Volumes/wor...|      EMMA|
|      Ava|  F|18053|dbfs:/Volumes/wor...|       AVA|
|  Madison|  F|17968|dbfs:/Volumes/wor...|   MADISON|
|   Sophia|  F|17031|dbfs:/Volumes/wor...|    SOPHIA|
|   Olivia|  F|16589|dbfs:/Volumes/wor...|    OLIVIA|
|  Abigail|  F|15476|dbfs:/Volumes/wor...|   ABIGAIL|
|   Hannah|  F|13323|dbfs:/Volumes/wor...|    HANNAH|
|Elizabeth|  F|13071|dbfs:/Volumes/wor...| ELIZABETH|
|  Addison|  F|11949|dbfs:/Volumes/wor...|   ADDISON|
| Samantha|  F|11883|dbfs:/Volumes/wor...|  SAMANTHA|
|   Ashley|  F|11428|dbfs:/Volumes/wor...|    ASHLEY|
|   Alyssa|  F|11275|dbfs:/Volumes/wor...|    ALYSSA|
|      Mia|  F|10922|dbfs:/Volumes/wor...|       MIA|
|    Chloe|  F|10759|dbfs:/V

In [0]:
# coditional column is a narrow transformation
df_flag = df_added.withColumn(
    "popularity",
    when(col("count") > 10000, "high").otherwise("medium")
)

df_flag.show(30)

+---------+---+-----+--------------------+----------+----------+
|     name|sex|count|         source_file|name_upper|popularity|
+---------+---+-----+--------------------+----------+----------+
|    Emily|  F|19367|dbfs:/Volumes/wor...|     EMILY|      high|
| Isabella|  F|19143|dbfs:/Volumes/wor...|  ISABELLA|      high|
|     Emma|  F|18390|dbfs:/Volumes/wor...|      EMMA|      high|
|      Ava|  F|18053|dbfs:/Volumes/wor...|       AVA|      high|
|  Madison|  F|17968|dbfs:/Volumes/wor...|   MADISON|      high|
|   Sophia|  F|17031|dbfs:/Volumes/wor...|    SOPHIA|      high|
|   Olivia|  F|16589|dbfs:/Volumes/wor...|    OLIVIA|      high|
|  Abigail|  F|15476|dbfs:/Volumes/wor...|   ABIGAIL|      high|
|   Hannah|  F|13323|dbfs:/Volumes/wor...|    HANNAH|      high|
|Elizabeth|  F|13071|dbfs:/Volumes/wor...| ELIZABETH|      high|
|  Addison|  F|11949|dbfs:/Volumes/wor...|   ADDISON|      high|
| Samantha|  F|11883|dbfs:/Volumes/wor...|  SAMANTHA|      high|
|   Ashley|  F|11428|dbfs

In [0]:
# drop column is a narrow transformation
df_dropped = df_flag.drop("source_file")

df_dropped.show()

+---------+---+-----+----------+----------+
|     name|sex|count|name_upper|popularity|
+---------+---+-----+----------+----------+
|    Emily|  F|19367|     EMILY|      high|
| Isabella|  F|19143|  ISABELLA|      high|
|     Emma|  F|18390|      EMMA|      high|
|      Ava|  F|18053|       AVA|      high|
|  Madison|  F|17968|   MADISON|      high|
|   Sophia|  F|17031|    SOPHIA|      high|
|   Olivia|  F|16589|    OLIVIA|      high|
|  Abigail|  F|15476|   ABIGAIL|      high|
|   Hannah|  F|13323|    HANNAH|      high|
|Elizabeth|  F|13071| ELIZABETH|      high|
|  Addison|  F|11949|   ADDISON|      high|
| Samantha|  F|11883|  SAMANTHA|      high|
|   Ashley|  F|11428|    ASHLEY|      high|
|   Alyssa|  F|11275|    ALYSSA|      high|
|      Mia|  F|10922|       MIA|      high|
|    Chloe|  F|10759|     CHLOE|      high|
|  Natalie|  F|10443|   NATALIE|      high|
|    Sarah|  F|10014|     SARAH|      high|
|   Alexis|  F| 9923|    ALEXIS|    medium|
|    Grace|  F| 9769|     GRACE|

In [0]:
# rename collumn is a narrow transformation
df_renamed = df_dropped.withColumnRenamed("name_upper", "NAME")

df_renamed.show()

+---------+---+-----+---------+----------+
|     name|sex|count|     NAME|popularity|
+---------+---+-----+---------+----------+
|    Emily|  F|19367|    EMILY|      high|
| Isabella|  F|19143| ISABELLA|      high|
|     Emma|  F|18390|     EMMA|      high|
|      Ava|  F|18053|      AVA|      high|
|  Madison|  F|17968|  MADISON|      high|
|   Sophia|  F|17031|   SOPHIA|      high|
|   Olivia|  F|16589|   OLIVIA|      high|
|  Abigail|  F|15476|  ABIGAIL|      high|
|   Hannah|  F|13323|   HANNAH|      high|
|Elizabeth|  F|13071|ELIZABETH|      high|
|  Addison|  F|11949|  ADDISON|      high|
| Samantha|  F|11883| SAMANTHA|      high|
|   Ashley|  F|11428|   ASHLEY|      high|
|   Alyssa|  F|11275|   ALYSSA|      high|
|      Mia|  F|10922|      MIA|      high|
|    Chloe|  F|10759|    CHLOE|      high|
|  Natalie|  F|10443|  NATALIE|      high|
|    Sarah|  F|10014|    SARAH|      high|
|   Alexis|  F| 9923|   ALEXIS|    medium|
|    Grace|  F| 9769|    GRACE|    medium|
+---------+

Wide transformations require data movement between partitions, making them more expensive. (they are costly)



In [0]:
# group by is a wide transformation
df_grouped = df.groupBy("sex").sum("count")

df_grouped.show()


+---+----------+
|sex|sum(count)|
+---+----------+
|  F|  16274582|
|  M|  17543206|
+---+----------+



In [0]:
# group by is a wide transformation
df_name_counts = df.groupBy("name").count()

df_name_counts.show()


+---------+-----+
|     name|count|
+---------+-----+
|    Emily|   18|
| Isabella|   18|
|     Emma|   18|
|      Ava|   18|
|  Madison|   18|
|   Sophia|   18|
|   Olivia|   18|
|  Abigail|   18|
|   Hannah|   18|
|Elizabeth|   18|
|  Addison|   18|
| Samantha|   17|
|   Ashley|   18|
|   Alyssa|   16|
|      Mia|   18|
|    Chloe|   18|
|  Natalie|   17|
|    Sarah|   17|
|   Alexis|   18|
|    Grace|   18|
+---------+-----+
only showing top 20 rows


Actions trigger execution of transformations and return results to the driver.

In [0]:
# examples with actions - count, show, collect

# Count (Action)
total_rows = df.count()
print(f"Total rows: {total_rows}")

# Show (Action)
df.show(5)

# Collect (Action)
sample = df.limit(5).collect()
print(sample)


Total rows: 306520
+--------+---+-----+--------------------+
|    name|sex|count|         source_file|
+--------+---+-----+--------------------+
|   Emily|  F|19367|dbfs:/Volumes/wor...|
|Isabella|  F|19143|dbfs:/Volumes/wor...|
|    Emma|  F|18390|dbfs:/Volumes/wor...|
|     Ava|  F|18053|dbfs:/Volumes/wor...|
| Madison|  F|17968|dbfs:/Volumes/wor...|
+--------+---+-----+--------------------+
only showing top 5 rows
[Row(name='Emily', sex='F', count=19367, source_file='dbfs:/Volumes/workspace/default/baby_names/yob2007.txt'), Row(name='Isabella', sex='F', count=19143, source_file='dbfs:/Volumes/workspace/default/baby_names/yob2007.txt'), Row(name='Emma', sex='F', count=18390, source_file='dbfs:/Volumes/workspace/default/baby_names/yob2007.txt'), Row(name='Ava', sex='F', count=18053, source_file='dbfs:/Volumes/workspace/default/baby_names/yob2007.txt'), Row(name='Madison', sex='F', count=17968, source_file='dbfs:/Volumes/workspace/default/baby_names/yob2007.txt')]


In [0]:
# possible silver layer transformation example 
df_silver_example = df.filter(col("count") > 200) \
              .groupBy("sex") \
              .agg({"count": "sum"}) \
              .withColumnRenamed("sum(count)", "total_count")

df_silver_example.write.mode("overwrite").saveAsTable("silver_baby_names_example")